<a href="https://colab.research.google.com/github/fianbio/AI-driven-PLA2-antivenome/blob/main/01_data_collection_PLA2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rdkit chembl_webresource_client pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/pla2'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

print('Data akan disimpan di:', DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data akan disimpan di: /content/drive/MyDrive/pla2/data


In [ ]:
import pandas as pd
from tqdm import tqdm
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

target_client = new_client.target
activity_client = new_client.activity
molecule_client = new_client.molecule

/usr/local/lib/python3.13/dist-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


In [ ]:
by_pref_name = pd.DataFrame(target_client.filter(pref_name__icontains='phospholipase A2'))
by_synonym = pd.DataFrame(target_client.filter(target_synonym__icontains='phospholipase A2'))

frames = [df for df in [by_pref_name, by_synonym] if not df.empty]
targets_df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['target_chembl_id']).reset_index(drop=True)

print(f'Dari pref_name: {len(by_pref_name)}')
print(f'Dari target_synonym: {len(by_synonym)}')
print(f'Total gabungan setelah hilangkan duplikat: {len(targets_df)}')

cols_to_show = [c for c in ['target_chembl_id', 'pref_name', 'organism', 'target_type'] if c in targets_df.columns]
targets_df[cols_to_show].head(30)

Dari pref_name: 51
Dari target_synonym: 63
Total gabungan setelah hilangkan duplikat: 64


,target_chembl_id,pref_name,organism,target_type
0,CHEMBL4715,"Phospholipase A2, major isoenzyme",Sus scrofa,SINGLE PROTEIN
1,CHEMBL4834,Cytosolic phospholipase A2 gamma,Homo sapiens,SINGLE PROTEIN
2,CHEMBL2097,Putative inactive group IIC secretory phosphol...,Homo sapiens,SINGLE PROTEIN
3,CHEMBL3816,Cytosolic phospholipase A2,Homo sapiens,SINGLE PROTEIN
4,CHEMBL3474,"Phospholipase A2, membrane associated",Homo sapiens,SINGLE PROTEIN
5,CHEMBL2154,Group IIE secretory phospholipase A2,Homo sapiens,SINGLE PROTEIN
6,CHEMBL4807,Phospholipase A2,Apis mellifera,SINGLE PROTEIN
7,CHEMBL4136,Cytosolic phospholipase A2 beta,Homo sapiens,SINGLE PROTEIN
8,CHEMBL4342,Group 10 secretory phospholipase A2,Homo sapiens,SINGLE PROTEIN
9,CHEMBL3213,85/88 kDa calcium-independent phospholipase A2,Homo sapiens,SINGLE PROTEIN


In [ ]:
relevant_targets = targets_df[
    (targets_df['target_type'] == 'SINGLE PROTEIN') &
    (targets_df['pref_name'].str.contains('phospholipase A2', case=False, na=False))
].copy()

print(f'{len(relevant_targets)} target terpilih setelah filter')
relevant_targets[cols_to_show]

49 target terpilih setelah filter


,target_chembl_id,pref_name,organism,target_type
0,CHEMBL4715,"Phospholipase A2, major isoenzyme",Sus scrofa,SINGLE PROTEIN
1,CHEMBL4834,Cytosolic phospholipase A2 gamma,Homo sapiens,SINGLE PROTEIN
2,CHEMBL2097,Putative inactive group IIC secretory phosphol...,Homo sapiens,SINGLE PROTEIN
3,CHEMBL3816,Cytosolic phospholipase A2,Homo sapiens,SINGLE PROTEIN
4,CHEMBL3474,"Phospholipase A2, membrane associated",Homo sapiens,SINGLE PROTEIN
5,CHEMBL2154,Group IIE secretory phospholipase A2,Homo sapiens,SINGLE PROTEIN
6,CHEMBL4807,Phospholipase A2,Apis mellifera,SINGLE PROTEIN
7,CHEMBL4136,Cytosolic phospholipase A2 beta,Homo sapiens,SINGLE PROTEIN
8,CHEMBL4342,Group 10 secretory phospholipase A2,Homo sapiens,SINGLE PROTEIN
9,CHEMBL3213,85/88 kDa calcium-independent phospholipase A2,Homo sapiens,SINGLE PROTEIN


In [ ]:
STANDARD_TYPES = ['IC50', 'Ki', 'EC50']

all_activities = []

for chembl_id in tqdm(relevant_targets['target_chembl_id'].tolist(), desc='Mengambil aktivitas per target'):
    res = activity_client.filter(
        target_chembl_id=chembl_id,
        standard_type__in=STANDARD_TYPES,
        standard_value__isnull=False,
    ).only([
        'molecule_chembl_id', 'canonical_smiles', 'standard_type',
        'standard_value', 'standard_units', 'target_chembl_id',
        'target_organism', 'assay_description'
    ])
    df_chunk = pd.DataFrame(res)
    if not df_chunk.empty:
        all_activities.append(df_chunk)

raw_activity_df = pd.concat(all_activities, ignore_index=True) if all_activities else pd.DataFrame()
print(f'Total baris data aktivitas mentah: {len(raw_activity_df)}')
raw_activity_df.head()

Mengambil aktivitas per target: 100%|██████████| 49/49 [06:06<00:00,  7.48s/it]

Total baris data aktivitas mentah: 2628


,assay_description,canonical_smiles,molecule_chembl_id,standard_type,standard_units,standard_value,target_chembl_id,target_organism,type,units,value
0,In vitro inhibition of porcine pancreatic phos...,CCCCCCCCCCCCCCCC(=O)N[C@@H](COP(=O)(O)OCCO)CC(C)C,CHEMBL101972,IC50,nM,30.0,CHEMBL4715,Sus scrofa,IC50,uM,0.03
1,In vitro inhibition of porcine pancreatic phos...,CCCCCCCCCCCCCCCC(=O)N[C@@H](COP(=O)(O)OCCO)CC(C)C,CHEMBL101972,IC50,nM,2600.0,CHEMBL4715,Sus scrofa,IC50,uM,2.6
2,In vitro inhibition of porcine pancreatic phos...,C[N+](C)(C)CCOP(=O)([O-])OCCNC(=O)c1ccc2ccccc2c1,CHEMBL99161,IC50,nM,8000.0,CHEMBL4715,Sus scrofa,IC50,uM,8.0
3,In vitro inhibition of porcine pancreatic phos...,C[N+](C)(C)CCOP(=O)([O-])OCCNC(=O)Cc1ccc2ccccc2c1,CHEMBL99160,IC50,nM,1000000.0,CHEMBL4715,Sus scrofa,IC50,uM,1000.0
4,In vitro inhibition of porcine pancreatic phos...,CCCCCCCCCCCCCCCC(=O)NCCOP(=O)([O-])OCC[N+](C)(C)C,CHEMBL101890,IC50,nM,4000.0,CHEMBL4715,Sus scrofa,IC50,uM,4.0


In [ ]:
df = raw_activity_df.copy()

df = df.dropna(subset=['canonical_smiles'])

df = df[df['standard_units'] == 'nM']
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df.dropna(subset=['standard_value'])
df = df[df['standard_value'] > 0]

print(f'Setelah filter satuan & nilai valid: {len(df)} baris')

Setelah filter satuan & nilai valid: 2620 baris


In [ ]:
def is_valid_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

tqdm.pandas(desc='Validasi SMILES')
df['smiles_valid'] = df['canonical_smiles'].progress_apply(is_valid_smiles)
df = df[df['smiles_valid']].drop(columns=['smiles_valid'])

print(f'Setelah validasi SMILES: {len(df)} baris')

Validasi SMILES: 100%|██████████| 2620/2620 [00:00<00:00, 4652.61it/s]

Setelah validasi SMILES: 2620 baris


In [ ]:
agg_df = (
    df.groupby(['molecule_chembl_id', 'canonical_smiles'], as_index=False)
    .agg(mean_standard_value=('standard_value', 'mean'), n_records=('standard_value', 'count'))
)

print(f'Jumlah molekul unik setelah agregasi: {len(agg_df)}')
agg_df.head()

Jumlah molekul unik setelah agregasi: 1596


,molecule_chembl_id,canonical_smiles,mean_standard_value,n_records
0,CHEMBL101015,CCCCCCCCCCCCCCCC(=O)OCC(COP(=O)(O)OCCO)NC(=O)C...,1765.0,2
1,CHEMBL101849,CCCCCCCCCCOc1cc(OCCCCCCCCCC)cc(N(CC(=O)O)CC(=O...,230.0,1
2,CHEMBL101890,CCCCCCCCCCCCCCCC(=O)NCCOP(=O)([O-])OCC[N+](C)(C)C,152000.0,2
3,CHEMBL101925,CC(C)C[C@H](COP(=O)([O-])OCC[N+](C)(C)C)NC(=O)...,160000.0,1
4,CHEMBL101972,CCCCCCCCCCCCCCCC(=O)N[C@@H](COP(=O)(O)OCCO)CC(C)C,1315.0,2


In [ ]:
STANDARD_TYPES = ['Inhibition']

all_activities = []

for chembl_id in tqdm(relevant_targets['target_chembl_id'].tolist(), desc='Mengambil aktivitas per target'):
    res = activity_client.filter(
        target_chembl_id=chembl_id,
        standard_type__in=STANDARD_TYPES,
        standard_value__isnull=False,
    ).only([
        'molecule_chembl_id', 'canonical_smiles', 'standard_type',
        'standard_value', 'standard_units', 'target_chembl_id',
        'target_organism', 'assay_description'
    ])
    df_chunk = pd.DataFrame(res)
    if not df_chunk.empty:
        all_activities.append(df_chunk)

raw_activity_inh = pd.concat(all_activities, ignore_index=True) if all_activities else pd.DataFrame()
print(f'Total baris data aktivitas mentah: {len(raw_activity_inh)}')
raw_activity_inh

Mengambil aktivitas per target: 100%|██████████| 49/49 [05:00<00:00,  6.14s/it]

Total baris data aktivitas mentah: 1002


,assay_description,canonical_smiles,molecule_chembl_id,standard_type,standard_units,standard_value,target_chembl_id,target_organism,type,units,value
0,In vitro inhibition of porcine pancreatic phos...,CCCCCCCCCCCCCCCC(=O)N[C@@H](COP(=O)(O)O)Cc1ccccc1,CHEMBL319067,Inhibition,%,42.0,CHEMBL4715,Sus scrofa,Inhibition,%,42.0
1,In vitro inhibition of porcine pancreatic phos...,CCCCCCCCCCCCCCCC(=O)N[C@@H](COP(=O)(O)O)CC(C)C,CHEMBL318193,Inhibition,%,22.0,CHEMBL4715,Sus scrofa,Inhibition,%,22.0
2,Percentage inhibitory activity against Phospho...,COc1ccc(/C(=N/O)c2cccc(Cl)c2)cc1OC,CHEMBL79195,Inhibition,%,85.3,CHEMBL4715,Sus scrofa,Inhibition,%,85.3
3,Percentage inhibitory activity against Phospho...,COc1ccc(/C(=N/O)c2cccc(Cl)c2)cc1OC,CHEMBL79195,Inhibition,%,90.2,CHEMBL4715,Sus scrofa,Inhibition,%,90.2
4,Percentage inhibitory activity against Phospho...,COc1ccc(/C(=N/O)c2cccc(Cl)c2)cc1OC,CHEMBL79195,Inhibition,%,72.4,CHEMBL4715,Sus scrofa,Inhibition,%,72.4
...,...,...,...,...,...,...,...,...,...,...,...
997,Inhibition of Pnpla8 in mouse Neuro2a cells at...,O=C(N1CCCCC1c1ccccc1)n1cc(-c2ccc(-c3cccc(CO)c3...,CHEMBL3263577,Inhibition,%,50.0,CHEMBL3259504,Mus musculus,INH,%,50.0
998,Inhibition of Pnpla8 in mouse Neuro2a cells at...,O=C(c1cccc(-c2ccc(-c3cn(C(=O)N4CCCCC4c4ccccc4)...,CHEMBL3263579,Inhibition,%,50.0,CHEMBL3259504,Mus musculus,INH,%,50.0
999,Inhibition of Pnpla8 in mouse Neuro2a cells at...,O=C(O)c1cccc(-c2ccc(-c3cn(C(=O)N4CCCCC4Cc4cccc...,CHEMBL3263582,Inhibition,%,50.0,CHEMBL3259504,Mus musculus,INH,%,50.0
1000,Inhibition of bee venom PLA2 at 4 uM,CC1(C)CCC[C@]2(C)[C@H]3C[C@@H](O)[C@]4(C)[C@H]...,CHEMBL477929,Inhibition,%,70.0,CHEMBL3885667,Apis mellifera,INH,%,70.0


In [ ]:
df_inh = raw_activity_inh.dropna(subset=['canonical_smiles']).copy()

df_inh = df_inh[df_inh['standard_units'] == '%'].copy()
df_inh['standard_value'] = pd.to_numeric(df_inh['standard_value'], errors='coerce')

# Filter rentang persentase valid (0 - 100%)
df_inh = df_inh.dropna(subset=['standard_value'])
df_inh = df_inh[(df_inh['standard_value'] >= 0) & (df_inh['standard_value'] <= 100)]

# Opsional: Label senyawa aktif (>= 50%)
df_inh['active_class'] = (df_inh['standard_value'] >= 50).astype(int)

print(f'Setelah filter data inhibition (%): {len(df_inh)} baris')

Setelah filter data inhibition (%): 999 baris


In [ ]:
def is_valid_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

tqdm.pandas(desc='Validasi SMILES')
df_inh['smiles_valid'] = df_inh['canonical_smiles'].progress_apply(is_valid_smiles)
df_inh = df_inh[df_inh['smiles_valid']].drop(columns=['smiles_valid'])

print(f'Setelah validasi SMILES: {len(df_inh)} baris')

Validasi SMILES: 100%|██████████| 999/999 [00:00<00:00, 5495.85it/s]

Setelah validasi SMILES: 999 baris


In [ ]:
agg_df_inh = (
    df_inh.groupby(['molecule_chembl_id', 'canonical_smiles'], as_index=False)
    .agg(mean_standard_value=('standard_value', 'mean'), n_records=('standard_value', 'count'))
)

print(f'Jumlah molekul unik setelah agregasi: {len(agg_df_inh)}')
agg_df_inh.head()

Jumlah molekul unik setelah agregasi: 434


,molecule_chembl_id,canonical_smiles,mean_standard_value,n_records
0,CHEMBL101,CCCCC1C(=O)N(c2ccccc2)N(c2ccccc2)C1=O,85.000000,1
1,CHEMBL1084102,O=C(O)CC[C@@H](Cc1ccc(OCc2ccccc2)cc1)NC(=O)CCC...,89.500000,2
2,CHEMBL109073,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC1=CCC(C2=CC(=O)OC...,65.415385,13
3,CHEMBL109301,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC1=CCC(C2=CC(=O)OC...,61.830769,13
4,CHEMBL1094086,O=C(CCCOc1ccccc1)C(F)(F)C(F)(F)F,99.100000,1


In [ ]:
ACTIVE_THRESHOLD_NM = 50

agg_df_inh['label'] = (agg_df_inh['mean_standard_value'] >= ACTIVE_THRESHOLD_NM).astype(int)

print('Distribusi label (1 = aktif, 0 = inaktif):')
print(agg_df_inh['label'].value_counts())

Distribusi label (1 = aktif, 0 = inaktif):
label
1    237
0    197
Name: count, dtype: int64


In [ ]:
print(len(agg_df))
print(len(agg_df_inh))

1596
434


In [ ]:
agg_df2 = pd.concat([agg_df, agg_df_inh], ignore_index=True)
len(agg_df2)

2030

In [ ]:
bioactivity_path = os.path.join(DATA_DIR, 'pla2_bioactivity_clean.csv')
agg_df2.to_csv(bioactivity_path, index=False)
print('Tersimpan di:', bioactivity_path)

Tersimpan di: /content/drive/MyDrive/pla2/data/pla2_bioactivity_clean.csv


In [ ]:
ACTIVE_THRESHOLD_NM = 1000

agg_df['label'] = (agg_df['mean_standard_value'] <= ACTIVE_THRESHOLD_NM).astype(int)

print('Distribusi label (1 = aktif, 0 = inaktif):')
print(agg_df['label'].value_counts())

Distribusi label (1 = aktif, 0 = inaktif):
label
0    1131
1     465
Name: count, dtype: int64


In [ ]:
bioactivity_path = os.path.join(DATA_DIR, 'pla2_bioactivity_clean.csv')
agg_df.to_csv(bioactivity_path, index=False)
print('Tersimpan di:', bioactivity_path)

Tersimpan di: /content/drive/MyDrive/pla2_drug_repurposing/data/pla2_bioactivity_clean.csv


In [ ]:
approved = molecule_client.filter(max_phase=4).only([
    'molecule_chembl_id', 'pref_name', 'molecule_structures'
])

approved_list = []
for m in tqdm(approved, desc='Mengambil approved drugs'):
    structures = m.get('molecule_structures')
    if structures and structures.get('canonical_smiles'):
        approved_list.append({
            'molecule_chembl_id': m['molecule_chembl_id'],
            'pref_name': m.get('pref_name'),
            'canonical_smiles': structures['canonical_smiles'],
        })

approved_df = pd.DataFrame(approved_list)
print(f'Total approved drugs dengan SMILES valid (mentah): {len(approved_df)}')

Mengambil approved drugs: 100%|██████████| 4225/4225 [05:31<00:00, 12.75it/s]

Total approved drugs dengan SMILES valid (mentah): 3417


In [ ]:
tqdm.pandas(desc='Validasi SMILES approved drugs')
approved_df['smiles_valid'] = approved_df['canonical_smiles'].progress_apply(is_valid_smiles)
approved_df = approved_df[approved_df['smiles_valid']].drop(columns=['smiles_valid'])
approved_df = approved_df.drop_duplicates(subset=['canonical_smiles']).reset_index(drop=True)

print(f'Total approved drugs final (valid & unik): {len(approved_df)}')
approved_df.head()

Validasi SMILES approved drugs: 100%|██████████| 3417/3417 [00:00<00:00, 4462.83it/s]


Total approved drugs final (valid & unik): 3417


,molecule_chembl_id,pref_name,canonical_smiles
0,CHEMBL2,PRAZOSIN,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC
1,CHEMBL3,NICOTINE,CN1CCC[C@H]1c1cccnc1
2,CHEMBL4,OFLOXACIN,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23
3,CHEMBL5,NALIDIXIC ACID,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21
4,CHEMBL6,INDOMETHACIN,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1


In [ ]:
approved_path = os.path.join(DATA_DIR, 'approved_drugs_chembl.csv')
approved_df.to_csv(approved_path, index=False)
print('Tersimpan di:', approved_path)

Tersimpan di: /content/drive/MyDrive/pla2/data/approved_drugs_chembl.csv


In [ ]:
targets_df

,cross_references,organism,pref_name,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Sus scrofa,"Phospholipase A2, major isoenzyme",False,CHEMBL4715,"[{'accession': 'P00592', 'component_descriptio...",SINGLE PROTEIN,9823
1,[],Homo sapiens,Cytosolic phospholipase A2 gamma,False,CHEMBL4834,"[{'accession': 'Q9UP65', 'component_descriptio...",SINGLE PROTEIN,9606
2,[],Homo sapiens,Putative inactive group IIC secretory phosphol...,False,CHEMBL2097,"[{'accession': 'Q5R387', 'component_descriptio...",SINGLE PROTEIN,9606
3,[],Homo sapiens,Cytosolic phospholipase A2,False,CHEMBL3816,"[{'accession': 'P47712', 'component_descriptio...",SINGLE PROTEIN,9606
4,[],Homo sapiens,"Phospholipase A2, membrane associated",False,CHEMBL3474,"[{'accession': 'P14555', 'component_descriptio...",SINGLE PROTEIN,9606
...,...,...,...,...,...,...,...,...
59,[],Homo sapiens,Patatin-like phospholipase domain-containing p...,False,CHEMBL3822353,"[{'accession': 'Q96AD5', 'component_descriptio...",SINGLE PROTEIN,9606
60,[],Homo sapiens,Phospholipase A and acyltransferase 3,False,CHEMBL3831244,"[{'accession': 'P53816', 'component_descriptio...",SINGLE PROTEIN,9606
61,[],Homo sapiens,Peroxiredoxin-6,False,CHEMBL4295741,"[{'accession': 'P30041', 'component_descriptio...",SINGLE PROTEIN,9606
62,[],Homo sapiens,COP1-AGTL,False,CHEMBL5483085,"[{'accession': 'Q96AD5', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606


In [ ]:
targets_df['organism'].unique()

array(['Sus scrofa', 'Homo sapiens', 'Apis mellifera',
       'Rattus norvegicus', 'Mus musculus', 'Naja melanoleuca',
       'Naja mossambica', 'Trimeresurus flavoviridis', 'Echis carinatus',
       'Oryctolagus cuniculus', 'Naja naja', 'Bos taurus',
       'Crotalus adamanteus', 'Bothropoides pauloensis',
       'Daboia russelii'], dtype=object)

In [ ]:
len(approved_df)

3417

In [ ]:
len(agg_df2)

2030